In [3]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict


import prompts

In [4]:
## prepare data

# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

#read qq data
qq_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv')
qq_ds = Dataset.from_pandas(qq_df.loc[:,['question','follow_up_questions']])

In [5]:
qq_df

,id,sample_id,question,follow_up_questions
0,43810e53-3082-47c6-9a1c-7d530035252d,-5742327688291876861,When does episode 40 of bunk'd come out?,### When does episode 42 of bunk'd come out?\n...
1,11255093-2327-4d39-89e2-337be043db4a,-3582047784487750233,Who won the ncaa football national championshi...,### Who won the 2016 season's ncaa football na...
2,02b27ffc-f65a-45c1-a4c8-c3a90ecb4e59,6811938153834854976,"As of 2015, when was the last time the death p...","### As of 2017, when was the last time the dea..."
3,6b3a0668-67f9-421d-9ea7-a74abfafe252,1700733897006170137,Where does backwards failure of the left ventr...,### Where does failure of the left ventricle c...
4,c5db51fb-8310-4a12-bde2-9a4e17615094,142117929623619257,Who won the Second Italo-Ethiopian War?,### Who won the First Italo-Ethiopian War?\n##...
...,...,...,...,...
2882,fd3c54ec-f2c6-4911-9da6-fde28becc139,7007176913425291748,Where is wynonna earp filming for season 1 sup...,"### Where is wynonna earp, the tv series story..."
2883,92a74567-8ff9-483c-b5be-f794572c9b4a,1770991828175209436,When were the first fortifications built for t...,### When were the first fortifications built f...
2884,4ac0d29e-0508-4d9e-b1e9-73b3378bcbfd,-1402493466405577008,Who sang a cover of What You Won't Do for Love...,### Who sang the original What You Won't Do Fo...
2885,bb6969dd-70f2-406e-9a0f-dc4dfdda8c2c,3611220285690789892,When is my friend dahmer movie coming out in l...,### When is my friend dahmer movie coming out ...


In [6]:
qq_ds

Dataset({
    features: ['question', 'follow_up_questions'],
    num_rows: 2887
})

In [7]:
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# calculate embeddings for all data
evidence_embeddings = model.encode(evidence_ls, convert_to_tensor=True).to(device)

In [8]:
# load generative model
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [9]:
importlib.reload(prompts)

<module 'prompts' from '/home/dataconv/deallab/djk/sf_rag/sf_rag/sf_rag/prompts.py'>

In [10]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [11]:
def evaluate_docs(query, docs):
    print(f"Query : {query}")
    print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
        
        attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        filter=(generated_text.split('\n')[0])
        print(filter)
        if '#relevant' in filter:
            outs.append((generated_text.split('\n')[1]).strip())
    
    return outs

In [12]:
for entry in qq_ds:
    query = entry['question']
    fu_questions = entry['follow_up_questions']

In [13]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=258)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [14]:
query = qq_ds[1]['question']
docs = retrieve_documents(query)
rel_docs = evaluate_docs(query, docs)
make_new_query(query, rel_docs)

tensor([122080,    340,    356, 151465,    384, 109975,  49036,    377,    350,
        158352], device='cuda:0')
Query : Who won the ncaa football national championship played in 2016?
----------------------------------------------------------------------------------------------------


/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:452: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


#irrelevant
#irrelevant
#irrelevant
#irrelevant
#relevant
#relevant
#irrelevant
#relevant
#irrelevant
#irrelevant


"['### Which team won the 2016 NCAA Football National Championship in a non-upset fashion?',\n    '### Who defeated Washington in the 2016 NCAA Football National Championship semifinals?',\n    '### Which team won the 2016 NCAA Football National Championship by a margin of 31 points?',\n    '### Which team played against Georgia in the 2016 NCAA Football National Championship and lost in overtime?']"

In [15]:
def preprocessing(new_questions):
    return list(((new_questions.split("['")[1]).split("']")[0]).split("',\n    '"))

In [16]:
preprocessing(make_new_query(query, rel_docs))

['### Who were the teams that made it to the semifinals of the 2016 NCAA Football National Championship?',
 '### Which team won the championship game of the 2016 NCAA Football National Championship?',
 '### Which team won the semifinals of the 2016 NCAA Football National Championship?',
 '### Which team won the 2016 NCAA Football National Championship and how did they win?',
 '### Which team lost the championship game of the 2016 NCAA Football National Championship?',
 '### Which team won the championship game in overtime in the 2016 NCAA Football National Championship?']

In [17]:
def make_new_answer(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['new_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ1']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=258)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [ ]:
sf_rag=dict()
perplexity_df=pd.DataFrame()

for i in range(5):
    query = qq_ds[i]['question']
    docs = retrieve_documents(query)
    rel_docs = evaluate_docs(query, docs)
    new_questions=preprocessing(make_new_query(query, rel_docs))
    print(new_questions)
    new_answers_list=[]
    for new_question in new_questions:
        new_docs=retrieve_documents(new_question)
        new_rel_docs=evaluate_docs(new_question, new_docs)
        if new_rel_docs:
            new_answers=make_new_answer(new_question, new_rel_docs)
            print(new_answers)
            new_answers_list.append(new_answers)
        else:
            print('All irrelevant docs')
        

In [130]:
from evaluate import load
perplexity = load("perplexity", module_type="metric")

In [162]:
sf_rag=dict()
perplexity_df=pd.DataFrame()

for i in range(5):
    query = qq_ds[i]['question']
    docs = retrieve_documents(query)
    rel_docs = evaluate_docs(query, docs)
    new_questions=preprocessing(make_new_query(query, rel_docs))
    print(new_questions)
    new_answers_list=[]
    for new_question in new_questions:
        new_docs=retrieve_documents(new_question)
        new_rel_docs=evaluate_docs(new_question, new_docs)
        if new_rel_docs:
            new_answers=make_new_answer(new_question, new_rel_docs)
            print(new_answers)
            new_answers_list.append(new_answers)
        else:
            print('All irrelevant docs')
    fq=[qq_df['follow_up_questions'][0]]
    nq=""
    for i in new_questions:
        nq+=i+'\n'
    tmp=list([nq])
    tmp.append(fq)
    results = perplexity.compute(predictions=tmp, model_id="meta-llama/Meta-Llama-3.1-8B-Instruct")
    print(results)
    perplexity_df['new_questions']=list([nq])
    perplexity_df['mean_perplexity']=results['mean_perplexity']
        
    

tensor([132,  53,  12, 179,   0, 115, 182,  36, 178, 264], device='cuda:0')
Query : When does episode 40 of bunk'd come out?
----------------------------------------------------------------------------------------------------
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#relevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
['### What is the release date of episode 40 of the TV series "Bunk\\\\\\\'d"?', '### How many episodes were in the TV series "Bunk\\\\\\\'d"?', '### What was the last episode date of the TV series "Bunk\\\\\\\'d"?', '### How long did the TV series "Bunk\\\\\\\'d" air for?']
tensor([ 57922,      0, 177344, 103117,  38090,  36238, 106761, 113501, 175969,
         66520], device='cuda:0')
Query : ### What is the release date of episode 40 of the TV series "Bunk\\\'d"?
----------------------------------------------------------------------------------------------------
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{'perplexities': [5.855372428894043, 12.548310279846191], 'mean_perplexity': 9.201841354370117}
tensor([122080,    340,    356, 151465,    384, 109975,  49036,    377,    350,
        158352], device='cuda:0')
Query : Who won the ncaa football national championship played in 2016?
----------------------------------------------------------------------------------------------------
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#relevant
#irrelevant
#relevant
#irrelevant
#irrelevant
['### Which teams played in the 2016 NCAA Football National Championship?', '### What was the score of the 2016 NCAA Football National Championship?', '### Which team was the runner-up in the 2016 NCAA Football National Championship?', '### How many times has Alabama won the NCAA Football National Championship?', '### What was the date of the 2016 NCAA Football National Championship?', '### Where was the 2016 NCAA Football National Championship held?']
tensor([122080, 151465, 158352, 158353, 1583

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

{'perplexities': [4.5435686111450195, 12.548290252685547], 'mean_perplexity': 8.545929431915283}
tensor([   615,    643,    653,    663,    673,    681, 148482, 148497, 148512,
        148527], device='cuda:0')
Query : Who won the Second Italo-Ethiopian War?
----------------------------------------------------------------------------------------------------
#irrelevant
#irrelevant
#irrelevant
#irrelevant
['### Which countries were involved in the Second Italo-Ethiopian War?', '### What were the key events or battles that took place during the Second Italo-Ethiopian War?', '### What was the outcome of the Second Italo-Ethiopian War, and how did it affect the region?', '### Were there any notable leaders or figures involved in the Second Italo-Ethiopian War?', '### What were the causes of the Second Italo-Ethiopian War, and how did it contribute to the broader context of colonialism and imperialism in Africa?']
tensor([   615,    643,    653,    663,    673,    681, 148482, 148497, 14851

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{'perplexities': [2.3158769607543945, 12.54830551147461], 'mean_perplexity': 7.432091236114502}


In [163]:
perplexity_df

,new_questions,mean_perplexity
0,### Which countries were involved in the Secon...,7.432091
